# 03 — Typed tools and tool choice
A tool is a bounded Python capability the model can request. First test the
plain function, then watch the CrewAI agent decide when to use its wrapper.



**Basic lesson 3 of 13.**


## Problem

The model cannot calculate from a trusted function unless that function is exposed.


## Concept

A CrewAI tool wraps a typed Python contract the agent may invoke.

CrewAI exposes a Python function through a schema and a natural-language description. The model sees the contract and can request a call; the runtime validates arguments and executes the function.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call   ◀ YOU ARE HERE
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
Crew Task ──▶ Agent ──▶ typed ROI tool request
                  ▲                    │
                  │               Python validation
                  │                    │
                  └──── observation ◀──┘
                 Agent ──▶ scenario answer + caveat
```

task → agent chooses ROI tool → validated inputs → scenario result. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Test the pure ROI function and its invalid-input contract before involving the model

Test the pure ROI function and its invalid-input contract before involving the model.


In [ ]:
import sys
import json
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

import workshop  # configure project-local CrewAI storage
from crewai import Agent, Crew, Process, Task
from crewai.tools import tool

from workshop.common import model
from workshop.tools import calculate_annual_roi

print("Pure function:", calculate_annual_roi(45_000, 30_000))
try:
    calculate_annual_roi(45_000, 0)
except ValueError as error:
    print("Input contract:", error)



## Wrap the tested function as a CrewAI tool
The docstring describes when the agent should use this capability. The
function still validates inputs in Python, independent of model behavior.



## Step 2 — Wrap that function with CrewAI's @tool decorator and write a precise description

Wrap that function with CrewAI's @tool decorator and write a precise description.


In [ ]:
@tool("Calculate annual ROI")
def roi_tool(annual_benefit: float, annual_cost: float) -> str:
    """Calculate a hypothetical annual ROI percentage from stated USD assumptions."""
    value = calculate_annual_roi(annual_benefit, annual_cost)
    return json.dumps({"roi_percent": value, "status": "scenario, not a guarantee"})


print("Available tool:", roi_tool.name)



## Give only the required tool to the agent
The calculation is a scenario, not a measured financial outcome.



## Step 3 — Give the agent only the needed tool, then distinguish the numeric tool result from the model's narrative

Give the agent only the needed tool, then distinguish the numeric tool result from the model's narrative.


In [ ]:
analyst = Agent(
    role="ROI Analyst",
    goal="Calculate transparent planning scenarios",
    backstory="You label estimates and assumptions.",
    llm=model(),
    tools=[roi_tool],
    max_iter=3,
    allow_delegation=False,
    verbose=True,
)
task = Task(
    description="Use the ROI tool for $45,000 annual benefit and $30,000 "
    "annual cost. Explain why the result is only a planning scenario.",
    expected_output="ROI percentage and assumption caveat.",
    agent=analyst,
)
print(Crew(agents=[analyst], tasks=[task], process=Process.sequential).kickoff().raw)



## Bonus scenario — a job posting
A second tool for a different job (inspired by CrewAI's *Job Posting* example).
The tool rejects an unknown level, and the agent sees that error as an observation.



In [ ]:
@tool("Salary band lookup")
def salary_band(level: str) -> str:
    """Return the fictional Northstar salary band in USD for a level: junior, mid, or senior."""
    bands = {"junior": "60k-80k", "mid": "80k-110k", "senior": "110k-150k"}
    if level.lower() not in bands:
        raise ValueError(f"unknown level '{level}'; use junior, mid, or senior")
    return json.dumps({"level": level.lower(), "band_usd": bands[level.lower()]})


recruiter = Agent(
    role="Recruiter",
    goal="Write accurate job ads",
    backstory="You never invent pay figures.",
    llm=model(),
    tools=[salary_band],
    max_iter=3,
    allow_delegation=False,
)
ad = Task(
    description="Write a two-sentence job ad for a senior agent engineer. "
    "Look up the salary band with the tool and include it.",
    expected_output="Two sentences that include the band from the tool.",
    agent=recruiter,
)
print(Crew(agents=[recruiter], tasks=[ad], process=Process.sequential).kickoff().raw)


## Failure mode to know about

The model selects tools from their names/descriptions, but Python must still reject invalid arguments. Test the pure function before involving the model.


## Production implication

Expose only needed tools, validate arguments inside Python, and limit permissions. Tool descriptions are part of the interface contract.


## Your turn

Try a zero annual cost in the pure function before the agent run.

**Checkpoint:** Invalid cost fails, and the model labels the valid ROI as a scenario.
